### Purpose
dbt-layer risk review on Postgres warehouse.

**Requires:** `make docker-up` + `dbt build`.

**Charts:** `notebooks/charts/warehouse/`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from notebooks.utils.data_loader import query_warehouse, warehouse_available, warehouse_count, warehouse_scalar
from notebooks.utils.style import C, MERCHANT_RISK_COLORS, RISK_BUCKET_COLORS, INR_CR, apply_theme, style_axes
from notebooks.utils.visualization import save_chart, new_fig

if not warehouse_available():
    raise ConnectionError("Start warehouse: make docker-up && make build")

apply_theme()
CHART_SUBDIR = "warehouse"

txn_n = warehouse_count("SELECT COUNT(*) AS n FROM intermediate.int_transactions_enriched")
print(f"Warehouse rows: {txn_n:,}")

### Chart - Risk model calibration

In [ ]:
risk = query_warehouse("SELECT fraud_risk_score, is_fraud FROM intermediate.int_transactions_enriched")
risk["fraud_risk_score"] = pd.to_numeric(risk["fraud_risk_score"])
risk["is_fraud"] = risk["is_fraud"].astype(bool)
risk["bucket"] = pd.cut(risk["fraud_risk_score"], [0, 30, 50, 70, 100], labels=["Low", "Med", "High", "Critical"], include_lowest=True)
bucket_order = ["Low", "Med", "High", "Critical"]
cal = risk.groupby("bucket", observed=True).agg(observed_fraud_pct=("is_fraud", "mean"), txns=("is_fraud", "count")).reindex(bucket_order)
fig, ax = new_fig(8, 4)
bars = ax.bar(
    bucket_order, (cal["observed_fraud_pct"] * 100).to_numpy(),
    color=[RISK_BUCKET_COLORS[str(b)] for b in bucket_order], width=0.55,
)
ax.set_ylabel("Observed fraud %")
ax.set_title("Model calibration: fraud % by score bucket", loc="left", fontweight="600")
save_chart(fig, "risk_score_calibration", CHART_SUBDIR)

`Monotonic rise Low→Critical proves the dbt score separates risk — High/Critical buckets are near-pure fraud.`

### Chart - Merchant volume vs fraud rate

In [ ]:
mer = query_warehouse("""
    SELECT
        COALESCE(SUM(f.amount), 0) AS total_volume,
        COUNT(f.transaction_id) AS total_transactions,
        ROUND(SUM(CASE WHEN f.is_fraud THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(f.transaction_id), 0), 2) AS fraud_rate_pct,
        CASE
            WHEN SUM(CASE WHEN f.is_fraud THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(f.transaction_id), 0) >= 5
                 OR AVG(f.fraud_risk_score) >= 60 THEN 'High Risk'
            WHEN SUM(CASE WHEN f.is_fraud THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(f.transaction_id), 0) >= 2
                 OR AVG(f.fraud_risk_score) >= 40 THEN 'Medium Risk'
            ELSE 'Low Risk'
        END AS merchant_risk_category
    FROM marts.dim_merchants m
    JOIN marts.fct_transactions f ON m.merchant_sk = f.merchant_sk
    GROUP BY m.merchant_id
    HAVING COUNT(f.transaction_id) >= 50
""")
mer["total_volume"] = pd.to_numeric(mer["total_volume"])
mer["fraud_rate_pct"] = pd.to_numeric(mer["fraud_rate_pct"])

fig, ax = new_fig(10, 5)
for cat, grp in mer.groupby("merchant_risk_category", sort=False):
    cat_label = str(cat)
    ax.scatter(
        grp["total_volume"], grp["fraud_rate_pct"],
        s=grp["total_transactions"] / 8, alpha=0.65,
        label=cat_label, color=MERCHANT_RISK_COLORS.get(cat_label, C["neutral"]),
        edgecolors="white", linewidths=0.4,
    )
ax.set_xlabel("Total volume (INR)")
ax.set_ylabel("Fraud rate (%)")
ax.xaxis.set_major_formatter(INR_CR)
ax.legend(title="Risk tier", fontsize=8, title_fontsize=8, loc="upper left")
ax.set_title("Merchant volume vs fraud rate", loc="left", fontweight="600")
style_axes(ax)
save_chart(fig, "merchant_risk_scatter", CHART_SUBDIR)

`High-risk merchants (red) cluster at higher volume and 16–20% fraud — bubble size = txn count.`

### Chart - Top merchants by fraud loss (₹)

In [ ]:
top_fraud = query_warehouse("""
    SELECT m.merchant_name, m.merchant_category,
           SUM(CASE WHEN f.is_fraud THEN f.amount ELSE 0 END) AS fraud_loss_inr
    FROM marts.dim_merchants m
    JOIN marts.fct_transactions f ON m.merchant_sk = f.merchant_sk
    GROUP BY m.merchant_name, m.merchant_category
    HAVING COUNT(f.transaction_id) >= 50 AND SUM(CASE WHEN f.is_fraud THEN 1 ELSE 0 END) > 0
    ORDER BY fraud_loss_inr DESC LIMIT 10
""")
top_fraud["fraud_loss_inr"] = pd.to_numeric(top_fraud["fraud_loss_inr"])
top_fraud = top_fraud.sort_values("fraud_loss_inr")
labels = [str(r.merchant_name)[:22] for r in top_fraud.itertuples(index=False)]
colors = [C["danger"] if i >= len(top_fraud) - 3 else C["primary"] for i in range(len(top_fraud))]

fig, ax = new_fig(10, 5)
bars = ax.barh(labels, top_fraud["fraud_loss_inr"].to_numpy(), color=colors, height=0.6)
ax.set_xlabel("Fraud loss (INR)")
ax.xaxis.set_major_formatter(INR_CR)
ax.set_title("Top 10 merchants by fraud loss", loc="left", fontweight="600")
style_axes(ax, grid_axis="x")
save_chart(fig, "top_fraud_merchants", CHART_SUBDIR)

`Top 3 highlighted in red — action list for merchant risk review.`

### Chart - Velocity rule uplift

In [ ]:
base = warehouse_scalar("SELECT AVG(CASE WHEN is_fraud THEN 1.0 ELSE 0 END)*100 AS rate FROM intermediate.int_transactions_enriched", "rate")
vel = warehouse_scalar("""
    SELECT AVG(CASE WHEN is_fraud THEN 1.0 ELSE 0 END)*100 AS rate
    FROM intermediate.int_transactions_enriched
    WHERE tx_count_1h >= 5 OR tx_count_24h >= 15
""", "rate")

fig, ax = new_fig(5.5, 4.5)
cats = ["Platform\nbaseline", "Velocity\nflagged"]
vals = [base, vel]
bars = ax.bar(cats, vals, color=[C["primary"], C["danger"]], width=0.42)
ax.set_ylim(0, max(vals) * 1.3)
ax.set_ylabel("Fraud rate (%)")
uplift_text = f"{vel / base:.1f}×" if base > 0 else "N/A"
ax.set_title(f"Velocity rule uplift ({uplift_text} vs baseline)", loc="left", fontweight="600")
style_axes(ax)
save_chart(fig, "velocity_fraud_uplift", CHART_SUBDIR)

`Flagged velocity txns run ~4.5× the platform fraud rate — confirms rule signal strength.`